# 05 — Weight Derivation

Derive dimension weights from logistic regression on historical site outcomes.

**Key challenge**: Class imbalance — ~50 successful sites vs ~15 blocked.
A model that always predicts "Successful" gets 77% accuracy for free.

**Four-layer defense**:
1. `class_weight='balanced'`
2. SMOTE oversampling
3. Stratified K-Fold
4. Macro F1 evaluation (not accuracy)

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import json
from pathlib import Path

In [ ]:
# Load validation dataset
from analytics.validation.validate import get_validation_dataset

validation = get_validation_dataset()
print(validation.groupby('label')['outcome'].value_counts())

In [ ]:
# Load features and match to validation set
features = pd.read_parquet('../../data/processed/features.parquet')
features['county_fips'] = features['county_fips'].astype(str).str.zfill(5)

merged = validation.merge(features, on='county_fips', how='inner')
print(f'Matched {len(merged)} validation sites to feature data')
print(f'  Successful: {(merged["label"]==1).sum()}')
print(f'  Blocked: {(merged["label"]==0).sum()}')

In [ ]:
# Derive weights
from analytics.scoring.weights import derive_weights, DIMENSION_COLS

weights = derive_weights(merged, merged['label'], DIMENSION_COLS)
print(f'\nFinal weights: {weights}')

In [ ]:
# Compare derived vs fallback
from analytics.scoring.weights import FALLBACK_WEIGHTS
import matplotlib.pyplot as plt

dims = list(weights.keys())
derived = [weights[d] for d in dims]
fallback = [FALLBACK_WEIGHTS[d] for d in dims]

x = range(len(dims))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - 0.15 for i in x], fallback, 0.3, label='Assumed', color='#6b7280')
ax.bar([i + 0.15 for i in x], derived, 0.3, label='Derived', color='#22c55e')
ax.set_xticks(x)
ax.set_xticklabels([d.title() for d in dims])
ax.set_ylabel('Weight')
ax.set_title('Assumed vs Data-Derived Dimension Weights')
ax.legend()
plt.tight_layout()
plt.show()